In [ ]:
!pip install -q openai-whisper cohere gTTS
!pip install -q openai-whisper cohere gTTS

In [ ]:
import whisper
import cohere
from gtts import gTTS
from IPython.display import Audio, display

print("Libraries imported successfully!")

In [ ]:
model = whisper.load_model("base")

print("Whisper model loaded successfully!")

In [ ]:
import getpass

COHERE_API_KEY = getpass.getpass("Enter your Cohere API Key: ")

co = cohere.ClientV2(api_key=COHERE_API_KEY)

print("Cohere connected successfully!")

In [ ]:
audio_file = record_audio("input.wav", 5)

print("Recording completed!")

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

def record_audio(filename="input.wav", seconds=5):
    display(Javascript("""
    async function recordAudio(seconds) {
        const stream = await navigator.mediaDevices.getUserMedia({audio: true});
        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();

        await new Promise(resolve => setTimeout(resolve, seconds * 1000));

        recorder.stop();

        await new Promise(resolve => {
            recorder.onstop = resolve;
        });

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {type: "audio/wav"});
        const reader = new FileReader();

        reader.readAsDataURL(blob);

        await new Promise(resolve => {
            reader.onloadend = resolve;
        });

        return reader.result;
    }
    """))

    data = eval_js(f"recordAudio({seconds})")
    audio = b64decode(data.split(",")[1])

    with open(filename, "wb") as f:
        f.write(audio)

    return filename

print("Recording function is ready!")

In [ ]:
audio_file = record_audio("input.wav", 10)

print("Recording completed!")

In [ ]:
result = model.transcribe("input.wav")

user_text = result["text"]

print("You said:")
print(user_text)

In [ ]:
response = co.chat(
    model="command-a-plus-05-2026",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI voice assistant. Answer clearly and briefly."
        },
        {
            "role": "user",
            "content": user_text
        }
    ]
)

assistant_text = response.message.content[0].text

print("AI Assistant:")
print(assistant_text)

In [ ]:
response = co.chat(
    model="command-a-plus-05-2026",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI voice assistant. Answer clearly and briefly."
        },
        {
            "role": "user",
            "content": user_text
        }
    ]
)

assistant_text = response.message.content[0].text

print("AI Assistant:")
print(assistant_text)

In [ ]:
assistant_text = ""

for item in response.message.content:
    if hasattr(item, "text"):
        assistant_text += item.text

print("AI Assistant:")
print(assistant_text)

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

tts = gTTS(text=assistant_text, lang="en")
tts.save("response.mp3")

print("Audio response created!")

In [ ]:
display(Audio("response.mp3", autoplay=True))